In [2]:
import pandas as pd
import numpy
import re 
import string
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
df = pd.read_csv('spam.csv',encoding='latin-1')
df['label']=df['label'].map({'ham':0,'spam':1})
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
all_words = []
for text in df['text']:
    all_words.extend(text.split())

words_count = Counter(all_words)
vocab_list = sorted(words_count, key=words_count.get, reverse=True)

vocab = {word: idx + 1 for idx, word in enumerate(vocab_list)}
print(f"vocab size: {len(vocab)}")
print(vocab.get('jack'))

vocab size: 15583
11974


In [6]:
def tokenize(text, vocab):
    return [vocab.get(word, 0) for word in text.split()]

df['encoded_text'] = df['text'].apply(lambda x: tokenize(x, vocab))

print("Original text:", df['text'][0])
print("Encoded text:", df['encoded_text'][0])

Original text: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Encoded text: [762, 421, 6317, 6318, 6319, 2842, 66, 7, 2258, 71, 146, 634, 3873, 157, 6320, 6321, 88, 53, 6322, 886]


In [7]:
class SpamDataset(Dataset):
    def __init__(self, texts, max_len):
        self.texts = texts
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        if len(text) < self.max_len:
            text = text + [0] * (self.max_len - len(text))
        else:
            text = text[:self.max_len]
        
        return torch.tensor(text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

train_dataset = SpamDataset(train_df['encoded_text'].tolist(), max_len=50)
test_dataset = SpamDataset(test_df['encoded_text'].tolist(), max_len=50)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 3900
Number of testing samples: 1672


In [9]:
class SpamClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size=50):
        super(SpamClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size + 1, embed_size, padding_idx=0)

        self.fc = nn.Linear(embed_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        
        sentence_embedded = embedded.mean(dim=1)

        out = self.fc(sentence_embedded)
        out = self.sigmoid(out)
        return out

model = SpamClassifier(vocab_size=len(vocab), embed_size=50).to(device)
print(model)

# Num of parms
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")

SpamClassifier(
  (embedding): Embedding(15584, 50, padding_idx=0)
  (fc): Linear(in_features=50, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Total trainable parameters: 779251
